# **Ensembl GraphQL Workshop (Proof-of-concept) Mini-notebook**

This mini-notebook demonstrates how to query the **Ensembl GraphQL API** programmatically using Python.


### **Learning Objective**

Retrieve gene and transcript data from Ensembl GraphQL and export into a tsv file.

### **Learning Outcome**

After completing this notebook, participants will be able to:

- Send a GraphQL query from Python
- Extract nested data (gene, transcripts)
- Export results to a TSV table
- Understand a simple debugging pattern for query errors.

### **Workshop Modules**

1. **Basic Biological Concepts**
   - Genes and what they represent in the genome
   - Transcripts and alternative splicing
   - Species and genome assemblies
Goal: This section provides a short conceptual overview so participants without a genomics background can interpret the biological entities they will query.

2. **Introduction**
   - What is Ensembl
   - How Ensembl organises biological data
   - Key entities in Ensembl (gene, transcripts, variants)
   - Why GraphQL was built

3. **GraphQL Basics**
   - Types and fields
   - Query structure
   - Nested queries

4. **Schema Exploration**
   - Navigating the GraphQL schema
   - Using the GraphQL interface

5. **Query Building**
   - Retrieving genes from symbols and IDs
   - Gene → transcript relationships
   - Retrieving genomic coordinates
   - Structured queries

6. **Exporting Data**
   - Parsing JSON responses
   - Converting nested data into tables
   - Exporting results to CSV / TSV.

7. **Debugging Queries**
   - Interpreting GraphQL error messages
   - Fixing broken queries

8. **AI-Assisted Querying**
   - Prompt templates for AI assistants

In [ ]:
import requests
import pandas as pd

endpoint = "https://beta.ensembl.org/data/graphql"

### **Helper Function**

This function:
- sends a GraphQL query to the Ensembl API and
- handles both HTTP and GraphQL errors.

In [ ]:
def run_query(query, variables=None):

    response = requests.post(
        endpoint,
        json={"query": query, "variables": variables}
    )

    if response.status_code != 200:
        raise Exception(f"HTTP error: {response.status_code}")

    result = response.json()

    if "errors" in result:
        raise Exception(f"GraphQL error: {result['errors']}")

    return result["data"]

### **Step 1 - Basic gene query**

We start by querying a single gene using its Ensembl stable ID.

In [ ]:
query_gene = """
query {
  gene(by_id: {
    genome_id: "a7335667-93e7-11ec-a39d-005056b38ce3",
    stable_id: "ENSG00000139618"
  }) {
    stable_id
    symbol
  }
}
"""

data = run_query(query_gene)
data

### **Step 2 - Retrieve Transcripts**

Now we retrieve transcript-level information under the same gene.

Genes contain multiple transcripts.

GraphQL allows nested queries to retrieve this hierarchy.

In [ ]:
query_transcripts = """
query {
  gene(by_id: {
    genome_id: "a7335667-93e7-11ec-a39d-005056b38ce3",
    stable_id: "ENSG00000139618"
  }) {
    symbol
    transcripts {
      stable_id
      symbol
      so_term
      version
    }
  }
}
"""

data = run_query(query_transcripts)
data

### **Step 3 - Convert Results to a Table**

In [ ]:
transcripts = data["gene"]["transcripts"]

records = []

for t in transcripts:
    records.append({
        "gene_symbol": data["gene"]["symbol"],
        "transcript_id": t["stable_id"],
        "transcript_symbol": t["symbol"],
        "type": t["so_term"],
        "version": t["version"]
    })

df = pd.DataFrame(records)
df

### **Step 4 - Export Results**

In [ ]:
df.to_csv("transcripts.tsv", sep="\t", index=False)

print("File exported: transcripts.tsv")

### **Step 5 - Batch Query Mode**

Researchers often need to retrieve data for multiple genes.

Here's a small example (for 2 genes) using the same pattern.

In [ ]:
genes = [
    "ENSG00000139618",
    "ENSG00000141510"
]

batch_records = []

for gene_id in genes:

    query = f"""
    query {{
      gene(by_id: {{
        genome_id: "a7335667-93e7-11ec-a39d-005056b38ce3",
        stable_id: "{gene_id}"
      }}) {{
        symbol
        transcripts {{
          stable_id
        }}
      }}
    }}
    """

    data = run_query(query)

    for t in data["gene"]["transcripts"]:
        batch_records.append({
            "gene": data["gene"]["symbol"],
            "transcript": t["stable_id"]
        })

batch_df = pd.DataFrame(batch_records)
batch_df

### **Step 6 - Debugging Example**

This example intentionally requests an invalid field to show how GraphQL errors look and how we catch them.

In [ ]:
bad_query = """
query {
  gene(by_id: {
    genome_id: "a7335667-93e7-11ec-a39d-005056b38ce3",
    stable_id: "ENSG00000139618"
  }) {
    wrong_field
  }
}
"""

try:
    run_query(bad_query)
except Exception as e:
    print("Error detected:")
    print(e)

### **Exercise 1**

Modify the query to **retrieve** transcripts for another gene of your choice.

---
---

**Hint**:

- Inspect the structure of previous queries.

- Reuse `step 2` and change only `stable_id`.

**Solution code here**:

<details>
<summary><b>Click to reveal solution</b></summary>

```python
tp53_query = """
query {
  gene(by_id: {
    genome_id: "a7335667-93e7-11ec-a39d-005056b38ce3",
    stable_id: "ENSG00000141510"
  }) {
    symbol
    transcripts {
      stable_id
    }
  }
}
"""

run_query(tp53_query)